# Model Training — GPU-Accelerated XGBoost & Ensemble Methods

**Course:** Accelerated Machine Learning  
**Notebook:** 2 of 3 — `ModelTraining.ipynb`  
**Dataset:** HIGGS  
**Date:** April 2026

---

## Pipeline Position

| Notebook | Role | Status |
|----------|------|--------|
| `DataAnalysis.ipynb` | EDA, feature selection | ✅ Complete |
| **`ModelTraining.ipynb`** | **Data preparation, training, tuning, serialisation** | **← This notebook** |
| `ModelEvaluation.ipynb` | Final test-set evaluation, visualisation, discussion | ⬜ Next |

## Scope of This Notebook

**Performs:**
- Reload full 11M-row HIGGS dataset from source
- Feature/label separation using the 27 features selected in DataAnalysis
- Stratified 70/15/15 train/validation/test split
- StandardScaler fitting (training set only)
- Baseline XGBoost reproduction (~0.738 Macro F1)
- Full-scale XGBoost training with GPU acceleration
- Hyperparameter tuning via random search (validation set)
- cuML Random Forest training (comparison model)
- Stacked XGBoost ensemble with out-of-fold meta-features
- Stratified 5-fold cross-validation of best model
- Model comparison and selection
- Serialisation of all artefacts for ModelEvaluation

**Deferred to `ModelEvaluation.ipynb`:**
- Evaluation on the held-out **test set**
- Confusion matrix, ROC curve, precision-recall curve
- Classification report and threshold analysis
- Extended discussion and conclusions

---

## 1. Imports & Environment Setup

All compute-heavy operations use the RAPIDS GPU stack (cuDF, cuML, CuPy) on a Quadro RTX 8000 (48 GB VRAM). CPU libraries (scikit-learn, joblib) are used only where GPU equivalents are unavailable — specifically for `StratifiedKFold`, `StandardScaler`, `LogisticRegression`, and metric computation.

**Known environment constraint:** cuML RandomForestClassifier in RAPIDS 23.08 trains successfully but does not expose `feature_importances_`. This was handled in DataAnalysis using a sklearn fallback; it does not affect this notebook since importance extraction is not repeated here.

In [27]:
import cudf
import cuml
import cupy as cp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
import pickle
import warnings
import joblib
import random

from cuml.ensemble import RandomForestClassifier as cuRF
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier

from cuml.preprocessing import StandardScaler
from cuml.linear_model import LogisticRegression
from cuml.metrics import accuracy_score

import gc

warnings.filterwarnings("ignore")

os.makedirs("artefacts", exist_ok=True)
os.makedirs("artefacts/base_models", exist_ok=True)
os.makedirs("plots", exist_ok=True)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f"cuDF   version : {cudf.__version__}")
print(f"cuML   version : {cuml.__version__}")
print(f"CuPy   version : {cp.__version__}")
print(f"NumPy  version : {np.__version__}")

mempool = cp.get_default_memory_pool()
print(f"\nGPU memory used at start : {mempool.used_bytes() / 1e9:.2f} GB")

cuDF   version : 23.08.00
cuML   version : 23.08.00
CuPy   version : 12.2.0
NumPy  version : 1.24.4

GPU memory used at start : 0.00 GB


---

## 2. Constants & Feature Definitions

The 27-feature set below was determined in `DataAnalysis.ipynb` through a three-stage pipeline:

1. **Variance threshold** (< 0.01): 0 features removed — all 28 pass
2. **Correlation filter** (|r| ≥ 0.80): `m_wwbb` removed — r = 0.896 with `m_wbb`
3. **RF Gini importance cross-validation**: full 27-feature set retained (F1 = 0.699) over 19-feature subset (F1 = 0.670)

No further feature engineering or selection is performed here. The feature set is treated as fixed input from the analysis stage.

In [2]:
DATA_PATH = "Partical.csv"
LABEL_COL = "label"

# Partical.csv has no header row — column names must be supplied explicitly.
# Order matches DataAnalysis.ipynb: label first, then all 28 features.
LOW_LEVEL = [
    'lepton_pT', 'lepton_eta', 'lepton_phi',
    'missing_energy_magnitude', 'missing_energy_phi',
    'jet_1_pt', 'jet_1_eta', 'jet_1_phi', 'jet_1_btag',
    'jet_2_pt', 'jet_2_eta', 'jet_2_phi', 'jet_2_btag',
    'jet_3_pt', 'jet_3_eta', 'jet_3_phi', 'jet_3_btag',
    'jet_4_pt', 'jet_4_eta', 'jet_4_phi', 'jet_4_btag',
]
HIGH_LEVEL = ['m_jj', 'm_jjj', 'm_lv', 'm_jlv', 'm_bb', 'm_wbb', 'm_wwbb']
ALL_FEATURES = LOW_LEVEL + HIGH_LEVEL
COLUMNS = [LABEL_COL] + ALL_FEATURES   # 1 label + 28 features = 29 columns

# 27 features used for modelling: m_wwbb excluded (|r|=0.896 with m_wbb, Stage 2 filter)
SELECTED_FEATURES = [f for f in ALL_FEATURES if f != 'm_wwbb']

print(f"Total CSV columns : {len(COLUMNS)} (label + {len(ALL_FEATURES)} features)")
print(f"Selected features : {len(SELECTED_FEATURES)} (m_wwbb excluded)")
print(f"Features          : {SELECTED_FEATURES}")

Total CSV columns : 29 (label + 28 features)
Selected features : 27 (m_wwbb excluded)
Features          : ['lepton_pT', 'lepton_eta', 'lepton_phi', 'missing_energy_magnitude', 'missing_energy_phi', 'jet_1_pt', 'jet_1_eta', 'jet_1_phi', 'jet_1_btag', 'jet_2_pt', 'jet_2_eta', 'jet_2_phi', 'jet_2_btag', 'jet_3_pt', 'jet_3_eta', 'jet_3_phi', 'jet_3_btag', 'jet_4_pt', 'jet_4_eta', 'jet_4_phi', 'jet_4_btag', 'm_jj', 'm_jjj', 'm_lv', 'm_jlv', 'm_bb', 'm_wbb']


### Column Name Definition & Initial Data Verification

`Partical.csv` has no header row — `cudf.read_csv` would assign integer column names (0, 1, 2, …) by default, causing a `KeyError` on any named column access. The full 29-column name list must therefore be supplied explicitly via `header=None, names=COLUMN_NAMES`.

Column order matches the original HIGGS dataset specification : label first, followed by 21 low-level kinematic features, then 7 high-level engineered invariant masses (including `m_wwbb` which is excluded from modelling but retained in the load to preserve column alignment).

This cell performs an initial load and schema verification — printing column names and dtypes to confirm the mapping is correct before the full pipeline proceeds.

In [3]:
COLUMN_NAMES = [
    "label",
    "lepton_pT", "lepton_eta", "lepton_phi",
    "missing_energy_magnitude", "missing_energy_phi",
    "jet_1_pt", "jet_1_eta", "jet_1_phi", "jet_1_btag",
    "jet_2_pt", "jet_2_eta", "jet_2_phi", "jet_2_btag",
    "jet_3_pt", "jet_3_eta", "jet_3_phi", "jet_3_btag",
    "jet_4_pt", "jet_4_eta", "jet_4_phi", "jet_4_btag",
    "m_jj", "m_jjj", "m_lv", "m_jlv", "m_bb", "m_wbb", "m_wwbb"
]

t0 = time.time()
df = cudf.read_csv(
    DATA_PATH,
    header=None,
    names=COLUMN_NAMES
)
load_time = time.time() - t0

df[LABEL_COL] = df[LABEL_COL].astype("int32")

print(f"Rows loaded  : {len(df):,}")
print(f"Columns      : {df.shape[1]}")
print(f"Load time    : {load_time:.2f}s")
print(f"\nColumn names : {list(df.columns)}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nGPU memory after load : {mempool.used_bytes() / 1e9:.2f} GB")

Rows loaded  : 11,000,000
Columns      : 29
Load time    : 15.47s

Column names : ['label', 'lepton_pT', 'lepton_eta', 'lepton_phi', 'missing_energy_magnitude', 'missing_energy_phi', 'jet_1_pt', 'jet_1_eta', 'jet_1_phi', 'jet_1_btag', 'jet_2_pt', 'jet_2_eta', 'jet_2_phi', 'jet_2_btag', 'jet_3_pt', 'jet_3_eta', 'jet_3_phi', 'jet_3_btag', 'jet_4_pt', 'jet_4_eta', 'jet_4_phi', 'jet_4_btag', 'm_jj', 'm_jjj', 'm_lv', 'm_jlv', 'm_bb', 'm_wbb', 'm_wwbb']

Dtypes:
label                         int32
lepton_pT                   float64
lepton_eta                  float64
lepton_phi                  float64
missing_energy_magnitude    float64
missing_energy_phi          float64
jet_1_pt                    float64
jet_1_eta                   float64
jet_1_phi                   float64
jet_1_btag                  float64
jet_2_pt                    float64
jet_2_eta                   float64
jet_2_phi                   float64
jet_2_btag                  float64
jet_3_pt                    float64

---

## 3. Data Loading

The full 11M-row HIGGS dataset is reloaded from source using `cudf.read_csv`. Each notebook in this pipeline loads data independently to maintain modularity — no inter-notebook state dependencies exist beyond the serialised artefacts produced by this notebook.

The label column is cast from `float32` to `int32`, consistent with the DataAnalysis notebook. This is required for cuML and XGBoost classifiers.

In [4]:
t0 = time.time()
df = cudf.read_csv(DATA_PATH, header=None, names=COLUMN_NAMES)
load_time = time.time() - t0

df[LABEL_COL] = df[LABEL_COL].astype("int32")

print(f"Rows loaded  : {len(df):,}")
print(f"Columns      : {df.shape[1]}")
print(f"Load time    : {load_time:.2f}s")
print(f"\nColumn names : {list(df.columns)}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nGPU memory after load : {mempool.used_bytes() / 1e9:.2f} GB")

Rows loaded  : 11,000,000
Columns      : 29
Load time    : 4.72s

Column names : ['label', 'lepton_pT', 'lepton_eta', 'lepton_phi', 'missing_energy_magnitude', 'missing_energy_phi', 'jet_1_pt', 'jet_1_eta', 'jet_1_phi', 'jet_1_btag', 'jet_2_pt', 'jet_2_eta', 'jet_2_phi', 'jet_2_btag', 'jet_3_pt', 'jet_3_eta', 'jet_3_phi', 'jet_3_btag', 'jet_4_pt', 'jet_4_eta', 'jet_4_phi', 'jet_4_btag', 'm_jj', 'm_jjj', 'm_lv', 'm_jlv', 'm_bb', 'm_wbb', 'm_wwbb']

Dtypes:
label                         int32
lepton_pT                   float64
lepton_eta                  float64
lepton_phi                  float64
missing_energy_magnitude    float64
missing_energy_phi          float64
jet_1_pt                    float64
jet_1_eta                   float64
jet_1_phi                   float64
jet_1_btag                  float64
jet_2_pt                    float64
jet_2_eta                   float64
jet_2_phi                   float64
jet_2_btag                  float64
jet_3_pt                    float64


---

## 4. Feature / Label Separation

X is constructed by selecting the 27 curated features from the cuDF DataFrame. y is the binary label column.

No transformations are applied at this stage. The class balance is confirmed before splitting to ensure stratification is correctly configured.

In [5]:
X = df[SELECTED_FEATURES]
y = df[LABEL_COL]

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

class_counts = y.value_counts().sort_index().to_pandas()
class_pct    = (class_counts / len(y) * 100).round(2)

print(f"\nClass distribution:")
for cls, cnt, pct in zip(class_counts.index, class_counts.values, class_pct.values):
    label = 'signal (1)' if cls == 1 else 'background (0)'
    print(f"  {label} : {cnt:,} ({pct:.1f}%)")

X shape : (11000000, 27)
y shape : (11000000,)

Class distribution:
  background (0) : 5,170,877 (47.0%)
  signal (1) : 5,829,123 (53.0%)


---

## 5. Stratified Train / Validation / Test Split

A **70 / 15 / 15** stratified split is applied, yielding approximately:
- **Training set** (~7.7M rows): used for all model fitting and cross-validation
- **Validation set** (~1.65M rows): used for hyperparameter tuning and model selection
- **Test set** (~1.65M rows): **LOCKED** — not used until `ModelEvaluation.ipynb`

Stratification preserves the ~53/47 signal/background ratio across all three splits, ensuring unbiased model selection. `sklearn.model_selection.train_test_split` is used because cuML's equivalent does not support stratification reliably for binary classification at this scale.

The two-stage split procedure is:
1. Hold out 30% as a temporary pool (val + test)
2. Split the temporary pool 50/50 into validation and test

>  **Test set isolation:** `X_test_np` and `y_test_np` are immediately serialised and must not be passed to any model, scaler fitting, or evaluation function in this notebook. All model selection uses `X_val_np` only.

In [6]:
# Convert to pandas for sklearn stratified split
X_pd = X.to_pandas()
y_pd = y.to_pandas()

# Stage 1: 70% train, 30% temp
X_train_pd, X_temp, y_train_pd, y_temp = train_test_split(
    X_pd, y_pd,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_pd
)

# Stage 2: 50/50 split of temp -> val and test
X_val_pd, X_test_pd, y_val_pd, y_test_pd = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

# Convert to numpy float32 / int32
X_train_np = X_train_pd.values.astype("float32")
y_train_np = y_train_pd.values.astype("int32")
X_val_np   = X_val_pd.values.astype("float32")
y_val_np   = y_val_pd.values.astype("int32")
X_test_np  = X_test_pd.values.astype("float32")
y_test_np  = y_test_pd.values.astype("int32")

print("Split sizes:")
print(f"  Train : {len(X_train_np):>10,} rows ({len(X_train_np)/len(df)*100:.1f}%)")
print(f"  Val   : {len(X_val_np):>10,} rows ({len(X_val_np)/len(df)*100:.1f}%)")
print(f"  Test  : {len(X_test_np):>10,} rows ({len(X_test_np)/len(df)*100:.1f}%)")

print("\nClass balance per split (signal %):")
for name, y_split in [("Train", y_train_np), ("Val", y_val_np), ("Test", y_test_np)]:
    sig_pct = y_split.sum() / len(y_split) * 100
    print(f"  {name:<5} : {sig_pct:.2f}% signal")

# Free cuDF memory — no longer needed
del X_pd, y_pd, X_temp, y_temp
del X_train_pd, X_val_pd, X_test_pd
del y_train_pd, y_val_pd, y_test_pd
del df, X, y
cp.get_default_memory_pool().free_all_blocks()
print(f"\nGPU memory after split : {mempool.used_bytes() / 1e9:.2f} GB")

Split sizes:
  Train :  7,700,000 rows (70.0%)
  Val   :  1,650,000 rows (15.0%)
  Test  :  1,650,000 rows (15.0%)

Class balance per split (signal %):
  Train : 52.99% signal
  Val   : 52.99% signal
  Test  : 52.99% signal

GPU memory after split : 0.00 GB


---

## 6. Feature Scaling

A `StandardScaler` is fitted **on the training set only** and applied to the validation and test sets. This prevents data leakage: fitting the scaler on all data would allow training distribution statistics to contaminate the validation and test splits, producing over-optimistic estimates.

**Note on tree models:** XGBoost and Random Forest are invariant to monotone feature transformations — scaling does not alter split points or predictions. Scaling is included here for two reasons:
1. The **LogisticRegression meta-model** in the stacking ensemble operates on predicted probabilities, but its inputs from different models may differ in scale; a consistent normalised representation aids convergence.
2. **Pipeline completeness:** the fitted scaler is serialised alongside the model so that ModelEvaluation can apply an identical transformation to the test set without re-fitting.

XGBoost and cuML RF receive the **unscaled** arrays (`X_train_np`, `X_val_np`). Only the stacking meta-model uses the scaled OOF features.

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_np)   # fit on train ONLY
X_val_scaled   = scaler.transform(X_val_np)          # transform, not fit
X_test_scaled  = scaler.transform(X_test_np)         # transform, not fit

print("StandardScaler fitted on training set only.")
print(f"\nPost-scaling statistics (first 3 features of X_train_scaled):")
print(f"  Mean : {X_train_scaled[:, :3].mean(axis=0).round(4)}")
print(f"  Std  : {X_train_scaled[:, :3].std(axis=0).round(4)}")
print("  (Expected: mean ≈ 0, std ≈ 1)")

StandardScaler fitted on training set only.

Post-scaling statistics (first 3 features of X_train_scaled):
  Mean : [ 0.  0. -0.]
  Std  : [1. 1. 1.]
  (Expected: mean ≈ 0, std ≈ 1)


---

## 7. Evaluation Helper

A reusable function for intermediate model evaluation. Metrics are computed on the **validation set only** throughout this notebook. Full final evaluation on the test set is deferred to `ModelEvaluation.ipynb`.

**Primary metric:** Macro F1-score. This weights precision and recall equally across both classes and is appropriate for a near-balanced binary classification problem where both signal detection and background rejection are equally important. Accuracy is not used as a primary metric because it is less informative for imbalanced distributions and does not distinguish false positive from false negative error types.

In [8]:
def evaluate_model(model, X, y_true, model_name, train_time=None, predict_fn=None):
    """
    Compute macro F1, precision, recall on validation set.
    predict_fn allows custom prediction logic (e.g. for ensemble).
    """
    if predict_fn is not None:
        y_pred = predict_fn(X)
    else:
        y_pred = model.predict(X)
    # Handle cuDF Series -> numpy
    if hasattr(y_pred, 'to_numpy'):
        y_pred = y_pred.to_numpy()
    # Handle CuPy array -> numpy
    elif hasattr(y_pred, 'get'):
        y_pred = y_pred.get()
    y_pred = np.asarray(y_pred).astype("int32")
    return {
        "Model":              model_name,
        "F1 (macro)":         round(f1_score(y_true, y_pred, average="macro"), 4),
        "Precision (macro)":  round(precision_score(y_true, y_pred, average="macro"), 4),
        "Recall (macro)":     round(recall_score(y_true, y_pred, average="macro"), 4),
        "Train Time (s)":     round(train_time, 1) if train_time is not None else None,
    }
results = []   # accumulate all model results here
print("evaluate_model() helper defined.")

evaluate_model() helper defined.


---

## 8. Baseline Model Reproduction

The DataAnalysis notebook established an XGBoost baseline of **F1 ≈ 0.738** using a 200,000-row sample with an 80/20 split. This result is reproduced here with identical hyperparameters to:

1. **Validate pipeline continuity** — confirm that data loading, feature selection, and split logic are correctly configured before scaling to 7.7M rows
2. **Establish a floor** — all subsequent models must demonstrably exceed this score to justify their additional complexity, following Occam's razor in model selection
3. **Document reproducibility** — slight variance (±0.02) is expected because the sample is drawn from the training partition (70% of data) rather than the full 11M rows, introducing minor distributional differences

The same hyperparameters as DataAnalysis are used: `n_estimators=100, max_depth=6, learning_rate=0.1, tree_method='hist'`.

In [9]:
print("=" * 55)
print(" Baseline Reproduction (200k sample, 80/20 split)")
print("=" * 55)

# Sample 200k from training set — mirrors DataAnalysis setup
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    len(X_train_np), size=200_000, replace=False
)
X_samp = X_train_np[sample_idx]
y_samp = y_train_np[sample_idx]

X_b_tr, X_b_val, y_b_tr, y_b_val = train_test_split(
    X_samp, y_samp,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_samp
)

model_baseline = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    verbosity=0
)

t0 = time.time()
model_baseline.fit(X_b_tr, y_b_tr)
baseline_train_time = time.time() - t0

baseline_result = evaluate_model(
    model_baseline, X_b_val, y_b_val,
    "XGBoost Baseline (200k, DataAnalysis params)",
    train_time=baseline_train_time
)
results.append(baseline_result)

print(f"Baseline F1 (macro) : {baseline_result['F1 (macro)']:.4f}")
print(f"Train time          : {baseline_train_time:.2f}s")
print(f"DataAnalysis target : ~0.738")
print(f"Difference          : {baseline_result['F1 (macro)'] - 0.738:+.4f}")

del X_samp, y_samp, X_b_tr, X_b_val, y_b_tr, y_b_val

 Baseline Reproduction (200k sample, 80/20 split)
Baseline F1 (macro) : 0.7168
Train time          : 5.68s
DataAnalysis target : ~0.738
Difference          : -0.0212


---

## 9. Full-Scale XGBoost Training

The model is now trained on the full ~7.7M-row training set using `tree_method='gpu_hist'`. This uses a GPU-accelerated histogram-based approximate split finding algorithm, providing near-exact gradient boosted tree construction with substantially reduced memory overhead compared to the exact algorithm.

**Hyperparameter choices and justifications:**

| Parameter | Value | Justification |
|-----------|-------|---------------|
| `n_estimators` | 500 | Larger datasets benefit from more boosting rounds; early stopping prevents over-iteration |
| `max_depth` | 8 | Deeper than the baseline (6) to capture higher-order feature interactions present in the HIGGS kinematic structure |
| `learning_rate` | 0.1 | Standard starting point for gradient boosting (Friedman, 2001); balanced convergence speed |
| `subsample` | 0.8 | Stochastic gradient boosting reduces variance and prevents overfitting (Friedman, 2002) |
| `colsample_bytree` | 0.8 | Feature subsampling per tree introduces diversity, analogous to Random Forest |
| `min_child_weight` | 5 | Minimum sum of instance weights per leaf; prevents over-specific splits on small subgroups |
| `gamma` | 0.1 | Minimum loss reduction for a split; acts as a regulariser on tree complexity |
| `reg_lambda` | 1.0 | L2 regularisation on leaf weights (XGBoost default); stabilises training |
| `early_stopping_rounds` | 50 | Halts training if validation logloss does not improve for 50 consecutive rounds |

In [10]:
print("=" * 55)
print(" Full-Scale XGBoost — Initial Configuration")
print("=" * 55)

xgb_initial = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="gpu_hist",
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    verbosity=0
)

t0 = time.time()
xgb_initial.fit(
    X_train_np, y_train_np,
    eval_set=[(X_val_np, y_val_np)],
    verbose=50
)
xgb_initial_time = time.time() - t0

xgb_initial_result = evaluate_model(
    xgb_initial, X_val_np, y_val_np,
    "XGBoost Initial (full 7.7M, gpu_hist)",
    train_time=xgb_initial_time
)
results.append(xgb_initial_result)

print(f"\nBest iteration  : {xgb_initial.best_iteration}")
print(f"F1 (macro)      : {xgb_initial_result['F1 (macro)']:.4f}")
print(f"Precision       : {xgb_initial_result['Precision (macro)']:.4f}")
print(f"Recall          : {xgb_initial_result['Recall (macro)']:.4f}")
print(f"Train time      : {xgb_initial_time:.1f}s")

 Full-Scale XGBoost — Initial Configuration
[0]	validation_0-logloss:0.67681
[50]	validation_0-logloss:0.54047
[100]	validation_0-logloss:0.52561
[150]	validation_0-logloss:0.51827
[200]	validation_0-logloss:0.51378
[250]	validation_0-logloss:0.51045
[300]	validation_0-logloss:0.50790
[350]	validation_0-logloss:0.50556
[400]	validation_0-logloss:0.50330
[450]	validation_0-logloss:0.50143
[499]	validation_0-logloss:0.49976

Best iteration  : 499
F1 (macro)      : 0.7493
Precision       : 0.7498
Recall          : 0.7490
Train time      : 29.6s


---

## 10. Hyperparameter Tuning — Random Search

Random search over the hyperparameter space is used in preference to grid search.  The random search explores the space more efficiently when not all hyperparameters are equally important — a property that holds for gradient boosted trees, where `max_depth` and `learning_rate` typically dominate.

**Search protocol:**
- 12 random configurations are drawn (budget ≈ 12 minutes GPU time)
- Each configuration is evaluated on `X_val_np` — the **validation set only**; the test set remains locked
- Evaluation uses Macro F1, consistent with the primary metric
- The best configuration is retrained on the full training set to produce `xgb_tuned`

In [11]:
print("=" * 55)
print(" Hyperparameter Tuning — Random Search (12 trials)")
print("=" * 55)

param_distributions = {
    "max_depth":         [4, 6, 8, 10, 12],
    "learning_rate":     [0.01, 0.05, 0.1, 0.2, 0.3],
    "n_estimators":      [200, 300, 500, 800],
    "subsample":         [0.6, 0.7, 0.8, 0.9],
    "colsample_bytree":  [0.6, 0.7, 0.8, 0.9],
    "min_child_weight":  [1, 3, 5, 10],
    "gamma":             [0, 0.1, 0.3, 0.5],
    "reg_lambda":        [0.5, 1.0, 2.0, 5.0],
}

N_TRIALS = 12
tuning_results = []

for trial in range(N_TRIALS):
    params = {k: random.choice(v) for k, v in param_distributions.items()}

    model = XGBClassifier(
        tree_method="gpu_hist",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        verbosity=0,
        **params
    )

    t0 = time.time()
    model.fit(X_train_np, y_train_np, verbose=0)
    t_elapsed = time.time() - t0

    y_pred = model.predict(X_val_np)
    f1 = f1_score(y_val_np, y_pred, average="macro")

    row = {**params, "f1": round(f1, 4), "train_time_s": round(t_elapsed, 1)}
    tuning_results.append(row)
    print(f"Trial {trial+1:>2}/{N_TRIALS} | F1={f1:.4f} | {t_elapsed:.0f}s | "
          f"depth={params['max_depth']} lr={params['learning_rate']}")

tuning_df = pd.DataFrame(tuning_results).sort_values("f1", ascending=False)
tuning_df.to_csv("artefacts/tuning_results.csv", index=False)

print(f"\nTop 5 configurations:")
print(tuning_df.head().to_string(index=False))

best_params = tuning_df.iloc[0].drop(["f1", "train_time_s"]).to_dict()
best_params = {k: (int(v) if isinstance(v, float) and v == int(v) else v)
               for k, v in best_params.items()}
print(f"\nBest hyperparameters: {best_params}")

 Hyperparameter Tuning — Random Search (12 trials)
Trial  1/12 | F1=0.6966 | 16s | depth=4 lr=0.01
Trial  2/12 | F1=0.7520 | 31s | depth=12 lr=0.2
Trial  3/12 | F1=0.7618 | 117s | depth=12 lr=0.05
Trial  4/12 | F1=0.7553 | 41s | depth=10 lr=0.1
Trial  5/12 | F1=0.7308 | 49s | depth=10 lr=0.01
Trial  6/12 | F1=0.7301 | 49s | depth=10 lr=0.01
Trial  7/12 | F1=0.7152 | 20s | depth=8 lr=0.01
Trial  8/12 | F1=0.7390 | 20s | depth=6 lr=0.1
Trial  9/12 | F1=0.7494 | 30s | depth=6 lr=0.2
Trial 10/12 | F1=0.7293 | 22s | depth=4 lr=0.1
Trial 11/12 | F1=0.7581 | 64s | depth=10 lr=0.2
Trial 12/12 | F1=0.7581 | 104s | depth=12 lr=0.2

Top 5 configurations:
 max_depth  learning_rate  n_estimators  subsample  colsample_bytree  min_child_weight  gamma  reg_lambda     f1  train_time_s
        12           0.05           800        0.7               0.9                 5    0.0         1.0 0.7618         117.0
        10           0.20           800        0.7               0.8                 3    0.1 

### 10b — Final Retraining with Best Hyperparameters

The random search identified the optimal configuration from 12 trials. That configuration is now retrained on the **full 7.7M-row training set** with early stopping against the validation set. This step is necessary because the random search trials were also trained on the full set (not a subsample), so the best trial's model is already well-fitted — but retraining with `early_stopping_rounds=50` allows the final model to terminate at the optimal number of trees rather than the fixed `n_estimators` ceiling used in the search.

**Why retrain rather than reuse the best trial's model?**

The random search loop did not use early stopping, so each trial trained for exactly `n_estimators` rounds. Retraining with early stopping ensures the final model stops at the point of minimum validation logloss, avoiding potential over-iteration that the fixed-budget search could not catch. This gives `xgb_tuned` a marginally better-calibrated number of boosting rounds than the raw search winner.

In [12]:
print("=" * 55)
print(" Retraining with Best Hyperparameters")
print("=" * 55)

xgb_tuned = XGBClassifier(
    tree_method="gpu_hist",
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    verbosity=0,
    **best_params
)

t0 = time.time()
xgb_tuned.fit(
    X_train_np, y_train_np,
    eval_set=[(X_val_np, y_val_np)],
    verbose=50
)
xgb_tuned_time = time.time() - t0

xgb_tuned_result = evaluate_model(
    xgb_tuned, X_val_np, y_val_np,
    "XGBoost Tuned (best random search params)",
    train_time=xgb_tuned_time
)
results.append(xgb_tuned_result)

print(f"\nTuned XGBoost — Validation Results:")
print(f"  F1 (macro) : {xgb_tuned_result['F1 (macro)']:.4f}")
print(f"  Precision  : {xgb_tuned_result['Precision (macro)']:.4f}")
print(f"  Recall     : {xgb_tuned_result['Recall (macro)']:.4f}")
print(f"  Train time : {xgb_tuned_time:.1f}s")

 Retraining with Best Hyperparameters
[0]	validation_0-logloss:0.68231
[50]	validation_0-logloss:0.53801
[100]	validation_0-logloss:0.51763
[150]	validation_0-logloss:0.50894
[200]	validation_0-logloss:0.50337
[250]	validation_0-logloss:0.49963
[300]	validation_0-logloss:0.49708
[350]	validation_0-logloss:0.49439
[400]	validation_0-logloss:0.49206
[450]	validation_0-logloss:0.48980
[500]	validation_0-logloss:0.48813
[550]	validation_0-logloss:0.48672
[600]	validation_0-logloss:0.48515
[650]	validation_0-logloss:0.48413
[700]	validation_0-logloss:0.48329
[750]	validation_0-logloss:0.48246
[799]	validation_0-logloss:0.48169

Tuned XGBoost — Validation Results:
  F1 (macro) : 0.7618
  Precision  : 0.7623
  Recall     : 0.7614
  Train time : 116.5s


---

## 11. cuML Random Forest — Comparison Model

A cuML Random Forest is trained as a comparison model. The GPU-accelerated cuML implementation trains on CuPy arrays directly on-device, avoiding CPU-GPU memory transfers during training.

**Known limitation (RAPIDS 23.08):** `feature_importances_` is not available from cuML RF in this environment. This was already handled in DataAnalysis using a sklearn fallback. Here, the RF is used purely as a trained classifier for evaluation via predictions — no importance extraction is performed.

**Hyperparameter rationale:** `max_depth=16` is deeper than the initial XGBoost (8) because RF uses full-depth trees by default in the ensemble context — individual trees overfit, but averaging reduces variance. `n_estimators=100` is a standard default; increasing it would improve the ensemble slightly at the cost of memory and training time.

In [13]:
print("=" * 55)
print(" cuML Random Forest — Comparison Model")
print("=" * 55)

X_train_cp = cp.asarray(X_train_np)
y_train_cp = cp.asarray(y_train_np).astype("int32")
X_val_cp   = cp.asarray(X_val_np)

rf_model = cuRF(
    n_estimators=100,
    max_depth=16,
    random_state=RANDOM_STATE,
    n_streams=4
)

t0 = time.time()
rf_model.fit(X_train_cp, y_train_cp)
rf_train_time = time.time() - t0

# cuML predict returns CuPy array — evaluate_model handles conversion
rf_result = evaluate_model(
    rf_model, X_val_cp, y_val_np,
    "cuML Random Forest (100 trees, depth 16)",
    train_time=rf_train_time
)
results.append(rf_result)

print(f"RF F1 (macro) : {rf_result['F1 (macro)']:.4f}")
print(f"Precision     : {rf_result['Precision (macro)']:.4f}")
print(f"Recall        : {rf_result['Recall (macro)']:.4f}")
print(f"Train time    : {rf_train_time:.1f}s")
print("\nNote: feature_importances_ unavailable in RAPIDS 23.08 — "
      "evaluation via predictions only.")

# Free GPU copies — no longer needed
del X_train_cp, y_train_cp, X_val_cp
cp.get_default_memory_pool().free_all_blocks()

 cuML Random Forest — Comparison Model
RF F1 (macro) : 0.7102
Precision     : 0.7122
Recall        : 0.7097
Train time    : 53.9s

Note: feature_importances_ unavailable in RAPIDS 23.08 — evaluation via predictions only.


---

## 12a. Homogenous Stacked XGBoost Ensemble

Model stacking trains a meta-learner on the predictions of multiple base models. This notebook implements a two-level architecture:

- **Level 0 (base models):** 3 XGBoost classifiers with deliberately diverse hyperparameter configurations — shallow/fast, medium, and deep/slow — to ensure the base models make different errors and can complement each other
- **Level 1 (meta-model):** Logistic Regression trained on the out-of-fold (OOF) predicted probabilities from all base models

### Why Out-of-Fold Predictions?

If base models generated predictions on the same data they were trained on, those predictions would contain information not available at inference time, causing the meta-model to learn from artificially confident (over-fitted) signals. This is **data leakage**.

OOF predictions avoid this: for each fold in the 5-fold cross-validation, the base model is trained on the remaining 4 folds and predicts only on the fold it never saw. Every training row therefore receives a prediction from a model that had no access to it during training — a faithful simulation of out-of-sample behaviour.

**Validation set handling:** The validation set is predicted by averaging predictions across all 5 folds for each base model. This produces the meta-features used to evaluate the ensemble on unseen data. The test set is not touched.

In [14]:
print("=" * 55)
print(" Stacked XGBoost Ensemble — OOF Meta-Feature Generation")
print("=" * 55)

base_configs = [
    # Shallow / fast — high bias, low variance
    {"max_depth": 4,  "learning_rate": 0.3,  "subsample": 0.7,
     "colsample_bytree": 0.7, "n_estimators": 300},
    # Medium — balanced profile
    {"max_depth": 8,  "learning_rate": 0.1,  "subsample": 0.8,
     "colsample_bytree": 0.8, "n_estimators": 500},
    # # Deep / slow — low bias, higher variance (Dropped as kernel kept dtying as memory was exceeded)
    # {"max_depth": 10, "learning_rate": 0.05, "subsample": 0.9,
    #  "colsample_bytree": 0.9, "n_estimators": 500},
]

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Meta-feature arrays
oof_preds = np.zeros((len(X_train_np), len(base_configs)), dtype="float32")
val_preds  = np.zeros((len(X_val_np),  len(base_configs)), dtype="float32")

base_models_fitted = []   # collect one fitted model per base config for serialisation

for i, config in enumerate(base_configs):
    print(f"\n--- Base model {i+1}/{len(base_configs)} | depth={config['max_depth']} "
          f"lr={config['learning_rate']} trees={config['n_estimators']} ---")
    fold_val_proba = np.zeros(len(X_val_np), dtype="float32")
    fold_oof_f1s   = []

    for fold, (tr_idx, oof_idx) in enumerate(skf.split(X_train_np, y_train_np)):
        m = XGBClassifier(
            tree_method="gpu_hist",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            verbosity=0,
            **config
        )
        m.fit(X_train_np[tr_idx], y_train_np[tr_idx], verbose=0)

        # OOF probabilities (class 1)
        oof_preds[oof_idx, i] = m.predict_proba(X_train_np[oof_idx])[:, 1]

        # Accumulate validation predictions (averaged across folds)
        fold_val_proba += m.predict_proba(X_val_np)[:, 1] / N_FOLDS

        fold_oof_f1 = f1_score(
            y_train_np[oof_idx],
            (oof_preds[oof_idx, i] > 0.5).astype("int32"),
            average="macro"
        )
        fold_oof_f1s.append(fold_oof_f1)
        print(f"  Fold {fold+1}: OOF F1 = {fold_oof_f1:.4f}")

        # Free GPU memory between folds
        del m
        cp.get_default_memory_pool().free_all_blocks()

    val_preds[:, i] = fold_val_proba
    overall_oof_f1 = f1_score(
        y_train_np, (oof_preds[:, i] > 0.5).astype("int32"), average="macro"
    )
    print(f"  Base model {i+1} OOF F1 (overall): {overall_oof_f1:.4f} "
          f"| mean per fold: {np.mean(fold_oof_f1s):.4f} ± {np.std(fold_oof_f1s):.4f}")

    # Keep one fold's model (last fold) for serialisation
    base_models_fitted.append(None)   # placeholder; full refit below

print("\nOOF meta-feature generation complete.")

 Stacked XGBoost Ensemble — OOF Meta-Feature Generation

--- Base model 1/2 | depth=4 lr=0.3 trees=300 ---
  Fold 1: OOF F1 = 0.7310
  Fold 2: OOF F1 = 0.7310
  Fold 3: OOF F1 = 0.7302
  Fold 4: OOF F1 = 0.7305
  Fold 5: OOF F1 = 0.7311
  Base model 1 OOF F1 (overall): 0.7307 | mean per fold: 0.7307 ± 0.0003

--- Base model 2/2 | depth=8 lr=0.1 trees=500 ---
  Fold 1: OOF F1 = 0.7495
  Fold 2: OOF F1 = 0.7486
  Fold 3: OOF F1 = 0.7491
  Fold 4: OOF F1 = 0.7492
  Fold 5: OOF F1 = 0.7495
  Base model 2 OOF F1 (overall): 0.7492 | mean per fold: 0.7492 ± 0.0003

OOF meta-feature generation complete.


### 12a — Meta-Model (Level 1) Training

With OOF meta-features generated, the second level of the stacking architecture can be trained. The input to the meta-model is `oof_preds` — an (N_train × 2) array where each column holds out-of-fold class-1 probabilities from one base model.

**Why LogisticRegression as the meta-learner?**

The meta-model's task is to learn how much weight to assign each base model's probability estimate. Logistic Regression is appropriate for three reasons:

1. **Bounded inputs.** OOF probabilities lie in [0, 1] — the feature space is inherently well-scaled and a linear model can exploit it directly without further non-linearity.
2. **Interpretability.** The learned coefficients directly reveal each base model's relative contribution. A coefficient near zero indicates the meta-model has discounted that base model's signal.
3. **Regularisation.** L2 penalty (C=1.0) prevents the meta-model from overfitting the OOF predictions, which are noisy estimates of out-of-sample probabilities.

A more complex meta-learner (e.g. another XGBoost) would likely overfit the small (N_train × 2) meta-feature matrix and generalise no better than LR.

**Scaling.** OOF probabilities from different base models may have different ranges depending on model calibration. A `StandardScaler` normalises these before fitting LR. The scaler is fitted on OOF predictions and applied identically to the validation meta-features — consistent with the train-only scaler fitting rule.

**GPU implementation.** cuML `LogisticRegression` is used in place of sklearn, keeping computation on-device and avoiding a large CPU-GPU transfer of the full OOF matrix. The fitted meta-model is serialised via joblib for use in ModelEvaluation.

In [15]:
# Clear unused GPU memory
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()
gc.collect()

print(f"GPU memory before meta-model: {cp.get_default_memory_pool().used_bytes() / 1e9:.2f} GB")

print("=" * 55)
print(" Stacking — GPU Meta-Model Training (cuML LogisticRegression)")
print("=" * 55)

# Convert OOF/meta arrays to GPU
oof_gpu = cudf.DataFrame(oof_preds)
val_gpu = cudf.DataFrame(val_preds)

y_train_gpu = cudf.Series(y_train_np.astype("int32"))
y_val_np = np.asarray(y_val_np).astype("int32")  # keep this CPU only for sklearn metrics

# Scale on GPU
meta_scaler = StandardScaler()
oof_scaled = meta_scaler.fit_transform(oof_gpu)
val_meta_scaled = meta_scaler.transform(val_gpu)

# GPU logistic regression
meta_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    penalty="l2",
    fit_intercept=True,
    verbose=0
)

t0 = time.time()
meta_model.fit(oof_scaled, y_train_gpu)
meta_train_time = time.time() - t0

# Predict on GPU
ensemble_pred_gpu = meta_model.predict(val_meta_scaled)

# Convert only final predictions to CPU for sklearn metrics
ensemble_pred = ensemble_pred_gpu.to_numpy().astype("int32")

ensemble_f1 = f1_score(y_val_np, ensemble_pred, average="macro")

ensemble_result = {
    "Model": "Stacked XGBoost Ensemble (3 base + cuML LR meta)",
    "F1 (macro)": round(ensemble_f1, 4),
    "Precision (macro)": round(precision_score(y_val_np, ensemble_pred, average="macro"), 4),
    "Recall (macro)": round(recall_score(y_val_np, ensemble_pred, average="macro"), 4),
    "Train Time (s)": round(meta_train_time, 2),
}

results.append(ensemble_result)

print(f"Ensemble F1 (macro) : {ensemble_f1:.4f}")
print(f"Precision           : {ensemble_result['Precision (macro)']:.4f}")
print(f"Recall              : {ensemble_result['Recall (macro)']:.4f}")
print(f"Train time          : {meta_train_time:.2f}s")
print(f"Meta-model coefficients: {meta_model.coef_}")

GPU memory before meta-model: 0.00 GB
 Stacking — GPU Meta-Model Training (cuML LogisticRegression)
Ensemble F1 (macro) : 0.7524
Precision           : 0.7529
Recall              : 0.7521
Train time          : 1.95s
Meta-model coefficients:           0        1
0 -0.376439  1.86242


---

## 12a. Heterogeneous Stacked Ensemble

The first stacked ensemble (Section 12a) used two XGBoost base models with different depths. While this provides some diversity through differing bias-variance profiles, both base models share the same algorithm family and therefore tend to make **correlated errors** on the same hard examples. When base model errors are correlated, the meta-learner cannot recover — it simply receives two similar probability estimates and can offer little improvement over the stronger individual model.

This section implements a **heterogeneous stacking** approach, using three fundamentally different algorithms as base models:

| Base Model | Algorithm | Inductive Bias |
|------------|-----------|----------------|
| Model 1 | XGBoost (depth=8, 300 trees) | Gradient boosting, captures non-linear interactions |
| Model 2 | cuML Random Forest (50 trees, depth=12) | Bagged trees, robust to outliers, high variance reduction |
| Model 3 | cuML Logistic Regression | Linear decision boundary, complementary to tree methods |

Each algorithm makes structurally different errors: XGBoost is sensitive to high-order feature interactions, RF is more robust to noisy features, and LR provides a stable linear baseline that generalises well near the decision boundary. The meta-learner (cuML LogisticRegression) learns the optimal combination of these diverse signals.

**Memory management:** cuML RF and LR require CuPy arrays. Aggressive cleanup (`del`, `free_all_blocks()`, `gc.collect()`) runs after every fold to prevent GPU OOM during the 15-fold loop (5 folds × 3 models).

In [20]:
print("=" * 60)
print("  Heterogeneous Stacked Ensemble — OOF Meta-Feature Generation")
print("=" * 60)

def get_class1_proba(proba):
    """Convert cuDF/CuPy/numpy model output to plain numpy float32 class-1 probabilities."""
    if hasattr(proba, 'values'):    # cuDF DataFrame/Series → CuPy
        proba = proba.values
    if hasattr(proba, 'get'):       # CuPy → numpy
        proba = proba.get()
    proba = np.asarray(proba, dtype="float32")
    if proba.ndim == 2:
        return proba[:, 1]
    return proba

N_FOLDS_HET  = 5
N_MODELS_HET = 3
skf_het      = StratifiedKFold(n_splits=N_FOLDS_HET, shuffle=True, random_state=RANDOM_STATE)
oof_preds_het = np.zeros((len(X_train_np), N_MODELS_HET), dtype="float32")
val_preds_het = np.zeros((len(X_val_np),   N_MODELS_HET), dtype="float32")

het_model_names = ["XGBoost (depth=8)", "cuML RF (50 trees)", "cuML LR"]

def make_het_base_model(i):
    if i == 0:
        return XGBClassifier(
            max_depth=8, learning_rate=0.1, n_estimators=300,
            subsample=0.8, colsample_bytree=0.8,
            tree_method="gpu_hist", eval_metric="logloss",
            random_state=RANDOM_STATE, verbosity=0
        )
    elif i == 1:
        return cuRF(n_estimators=50, max_depth=12,
                    random_state=RANDOM_STATE, n_streams=1)
    else:
        from cuml.linear_model import LogisticRegression as cuLR
        return cuLR(C=1.0, penalty="l2", max_iter=1000)

for i in range(N_MODELS_HET):
    print(f"\n── Base model {i+1}/{N_MODELS_HET}: {het_model_names[i]} ──")
    fold_val = np.zeros(len(X_val_np), dtype="float32")

    for fold, (tr_idx, oof_idx) in enumerate(skf_het.split(X_train_np, y_train_np)):
        print(f"   fold {fold+1}/{N_FOLDS_HET} ...", end=" ", flush=True)
        t0 = time.time()
        m = make_het_base_model(i)

        if i == 0:                              # XGBoost — numpy input
            m.fit(X_train_np[tr_idx], y_train_np[tr_idx], verbose=0)
            oof_preds_het[oof_idx, i] = get_class1_proba(m.predict_proba(X_train_np[oof_idx]))
            fold_val                  += get_class1_proba(m.predict_proba(X_val_np)) / N_FOLDS_HET
        else:                                   # cuML — CuPy input
            X_tr_cp  = cp.asarray(X_train_np[tr_idx])
            y_tr_cp  = cp.asarray(y_train_np[tr_idx].astype("float32"))
            X_oof_cp = cp.asarray(X_train_np[oof_idx])
            X_val_cp = cp.asarray(X_val_np)
            m.fit(X_tr_cp, y_tr_cp)
            oof_preds_het[oof_idx, i] = get_class1_proba(m.predict_proba(X_oof_cp))
            fold_val                  += get_class1_proba(m.predict_proba(X_val_cp)) / N_FOLDS_HET
            del X_tr_cp, y_tr_cp, X_oof_cp, X_val_cp

        del m
        cp.get_default_memory_pool().free_all_blocks()
        gc.collect()
        print(f"{time.time()-t0:.1f}s")

    val_preds_het[:, i] = fold_val
    overall_oof_f1 = f1_score(
        y_train_np, (oof_preds_het[:, i] > 0.5).astype("int32"), average="macro"
    )
    print(f"   ✓ model {i+1} OOF F1 (overall): {overall_oof_f1:.4f}")

print("\nHeterogeneous OOF generation complete.")
print(f"  oof_preds_het shape : {oof_preds_het.shape}")
print(f"  val_preds_het shape : {val_preds_het.shape}")

  Heterogeneous Stacked Ensemble — OOF Meta-Feature Generation

── Base model 1/3: XGBoost (depth=8) ──
   fold 1/5 ... 17.7s
   fold 2/5 ... 17.5s
   fold 3/5 ... 17.3s
   fold 4/5 ... 17.2s
   fold 5/5 ... 17.2s
   ✓ model 1 OOF F1 (overall): 0.7435

── Base model 2/3: cuML RF (50 trees) ──
   fold 1/5 ... 25.7s
   fold 2/5 ... 24.2s
   fold 3/5 ... 24.1s
   fold 4/5 ... 24.2s
   fold 5/5 ... 24.1s
   ✓ model 2 OOF F1 (overall): 0.6948

── Base model 3/3: cuML LR ──
   fold 1/5 ... 2.8s
   fold 2/5 ... [W] [01:32:48.550552] L-BFGS stopped, because the line search failed to advance (step delta = 0.000000)
2.1s
   fold 3/5 ... [W] [01:32:50.573454] L-BFGS stopped, because the line search failed to advance (step delta = 0.000000)
2.0s
   fold 4/5 ... 2.8s
   fold 5/5 ... 1.7s
   ✓ model 3 OOF F1 (overall): 0.6047

Heterogeneous OOF generation complete.
  oof_preds_het shape : (7700000, 3)
  val_preds_het shape : (1650000, 3)


### 12b — Heterogeneous Meta-Model Training

With heterogeneous OOF features generated, the meta-model is fitted following the same procedure as Section 12a. The input is now a (N_train × 3) matrix: column 0 (XGBoost probabilities), column 1 (cuML RF probabilities), column 2 (cuML LR probabilities). Each column represents a structurally different view of each training example.

The meta-learner must learn to combine qualitatively different probability signals. Inspection of the fitted coefficients reveals the relative trust assigned to each base model's output. Where the RF or LR provides complementary information (e.g. correctly classifying examples the XGBoost base model scores poorly), the meta-model can up-weight those columns accordingly.

**Observed result.** The heterogeneous ensemble (F1=0.7449) underperformed both the homogeneous stack (F1=0.7524) and the tuned XGBoost (F1=0.7618). This is consistent with theoretical expectations: ensemble diversity requires not only algorithmic difference but also *adequate base model strength*. The cuML RF (50 trees) and cuML LR base models perform near 0.71 and below individually — introducing noise that the meta-model cannot fully compensate for. This finding reinforces the parsimony argument for the single tuned XGBoost as the final model.

In [30]:
cp.get_default_memory_pool().free_all_blocks()
gc.collect()

print("=" * 60)
print("  Heterogeneous Stack — GPU Meta-Model Training")
print("=" * 60)

oof_het_gpu     = cudf.DataFrame(oof_preds_het.astype("float32"))
val_het_gpu     = cudf.DataFrame(val_preds_het.astype("float32"))
y_train_gpu_het = cudf.Series(y_train_np.astype("int32"))

meta_scaler_het = StandardScaler()
oof_het_scaled  = meta_scaler_het.fit_transform(oof_het_gpu)
val_het_scaled  = meta_scaler_het.transform(val_het_gpu)

meta_model_het = LogisticRegression(C=1.0, penalty="l2", max_iter=1000)

t0 = time.time()
meta_model_het.fit(oof_het_scaled, y_train_gpu_het)
het_meta_time = time.time() - t0

het_pred_gpu = meta_model_het.predict(val_het_scaled)
if hasattr(het_pred_gpu, 'to_numpy'):
    het_pred = het_pred_gpu.to_numpy()
elif hasattr(het_pred_gpu, 'get'):
    het_pred = het_pred_gpu.get()
het_pred = np.asarray(het_pred).astype("int32")

het_f1   = f1_score(y_val_np, het_pred, average="macro")
het_prec = precision_score(y_val_np, het_pred, average="macro")
het_rec  = recall_score(y_val_np, het_pred, average="macro")

het_ensemble_result = {
    "Model":              "Heterogeneous Stack (XGB + RF + LR → cuML LR meta)",
    "F1 (macro)":         round(het_f1, 4),
    "Precision (macro)":  round(het_prec, 4),
    "Recall (macro)":     round(het_rec, 4),
    "Train Time (s)":     round(het_meta_time, 2),
}
results.append(het_ensemble_result)

print(f"Heterogeneous Ensemble F1  : {het_f1:.4f}")
print(f"Precision                  : {het_prec:.4f}")
print(f"Recall                     : {het_rec:.4f}")
print(f"Meta-model fit time        : {het_meta_time:.2f}s")
print(f"Meta-model coefficients    : {meta_model_het.coef_}")

  Heterogeneous Stack — GPU Meta-Model Training
Heterogeneous Ensemble F1  : 0.7449
Precision                  : 0.7454
Recall                     : 0.7446
Meta-model fit time        : 0.08s
Meta-model coefficients    :           0         1         2
0  1.544353 -0.102138  0.011255


### Interim Comparison — Pre-CV Snapshot

This cell deduplicates the `results` list (guarding against cells being re-run in an interactive session) and produces an early comparison table before cross-validation runs. This snapshot shows the ranking of all models trained so far — baseline, initial XGBoost, tuned XGBoost, cuML RF, and both stacking ensembles — evaluated on the validation set.

> **Note:** `cv_mean` and `cv_std` are not yet computed at this point; the CV column will appear as `None` for all models in this snapshot. The definitive comparison including CV results is produced in Section 14 after cross-validation completes.

In [31]:
# Deduplicate in case any cell was re-run
seen, unique_results = set(), []
for r in results:
    if r["Model"] not in seen:
        seen.add(r["Model"])
        unique_results.append(r)
results = unique_results

print("=" * 65)
print(" Model Comparison — Validation Set Results")
print("=" * 65)

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values("F1 (macro)", ascending=False).reset_index(drop=True)

# Add CV result for tuned XGBoost
cv_label = f"{cv_mean:.4f} ± {cv_std:.4f}"
comparison_df["CV F1 (5-fold)"] = comparison_df["Model"].apply(
    lambda m: cv_label if "Tuned" in m else None
)

print(comparison_df.to_string(index=False))

comparison_df.to_csv("artefacts/model_comparison.csv", index=False)
print("\nComparison table saved to artefacts/model_comparison.csv")

# --- Model selection logic ---
tuned_f1_row   = comparison_df[comparison_df["Model"].str.contains("Tuned")].iloc[0]
ensemble_rows  = comparison_df[comparison_df["Model"].str.contains("Stack")]

best_ensemble_f1 = ensemble_rows["F1 (macro)"].max() if not ensemble_rows.empty else 0.0
delta = best_ensemble_f1 - tuned_f1_row["F1 (macro)"]

if not ensemble_rows.empty and delta >= 0.005:
    best_ens_row = ensemble_rows.loc[ensemble_rows["F1 (macro)"].idxmax()]
    SELECTED_MODEL_NAME = "Heterogeneous Stack" if "Heterogeneous" in best_ens_row["Model"] else "Homogeneous Stack"
    best_model = None
    print(f"\nSelected: {best_ens_row['Model']} (Δ={delta:+.4f} over tuned XGBoost — exceeds 0.5% threshold)")
else:
    SELECTED_MODEL_NAME = "Tuned XGBoost"
    best_model = xgb_tuned
    print(f"\nSelected: Tuned XGBoost (best ensemble Δ={delta:+.4f} — below 0.5% threshold; prefer parsimony)")

print(f"Selected model : {SELECTED_MODEL_NAME}")
print(f"Validation F1  : {tuned_f1_row['F1 (macro)']:.4f}")
print(f"CV F1 (5-fold) : {cv_mean:.4f} ± {cv_std:.4f}")

 Model Comparison — Validation Set Results
                                             Model  F1 (macro)  Precision (macro)  Recall (macro)  Train Time (s)  CV F1 (5-fold)
         XGBoost Tuned (best random search params)      0.7618             0.7623          0.7614          116.50 0.7606 ± 0.0002
  Stacked XGBoost Ensemble (3 base + cuML LR meta)      0.7524             0.7529          0.7521            1.95            None
             XGBoost Initial (full 7.7M, gpu_hist)      0.7493             0.7498          0.7490           29.60            None
Heterogeneous Stack (XGB + RF + LR → cuML LR meta)      0.7449             0.7454          0.7446            0.08            None
      XGBoost Baseline (200k, DataAnalysis params)      0.7168             0.7169          0.7168            5.70            None
          cuML Random Forest (100 trees, depth 16)      0.7102             0.7122          0.7097           53.90            None

Comparison table saved to artefacts/model_comp

---

## 13. Cross-Validation of Best Model

Stratified 5-fold cross-validation is run on the training set using the best-performing model identified so far. Cross-validation provides a more robust generalisation estimate than a single validation split because it averages performance across five disjoint holdout folds, reducing sensitivity to any particular random partition.

**Important:** CV operates on `X_train_np` only. The validation set is not involved. This section runs after model comparison to apply CV only to the selected winner, avoiding unnecessary computation.

In [26]:
print("=" * 55)
print(" Cross-Validation — Tuned XGBoost (5-fold, train set)")
print("=" * 55)
print("Running CV on the tuned XGBoost (best random search config).")
print("CV runs on training set only — validation set not used.\n")

cv_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = []

for fold, (tr_idx, val_idx) in enumerate(cv_skf.split(X_train_np, y_train_np)):
    cv_model = XGBClassifier(
        tree_method="gpu_hist",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        verbosity=0,
        **best_params
    )
    cv_model.fit(X_train_np[tr_idx], y_train_np[tr_idx], verbose=0)
    y_cv_pred = cv_model.predict(X_train_np[val_idx])
    fold_f1 = f1_score(y_train_np[val_idx], y_cv_pred, average="macro")
    cv_scores.append(fold_f1)
    print(f"  Fold {fold+1}: F1 = {fold_f1:.4f}")

    del cv_model
    cp.get_default_memory_pool().free_all_blocks()

cv_mean = np.mean(cv_scores)
cv_std  = np.std(cv_scores)

print(f"\nCV F1 (macro): {cv_mean:.4f} ± {cv_std:.4f}")

cv_results_df = pd.DataFrame({"fold": range(1, 6), "f1_macro": cv_scores})
cv_results_df.to_csv("artefacts/cv_results.csv", index=False)
print("CV results saved to artefacts/cv_results.csv")

 Cross-Validation — Tuned XGBoost (5-fold, train set)
Running CV on the tuned XGBoost (best random search config).
CV runs on training set only — validation set not used.

  Fold 1: F1 = 0.7607
  Fold 2: F1 = 0.7604
  Fold 3: F1 = 0.7604
  Fold 4: F1 = 0.7607
  Fold 5: F1 = 0.7609

CV F1 (macro): 0.7606 ± 0.0002
CV results saved to artefacts/cv_results.csv


---

## 14. Model Comparison & Selection

All trained models are compared on the validation set using Macro F1 as the primary criterion. The comparison is for **model selection purposes only** — the validation set is a proxy for generalisation, not a final performance estimate.

**Selection rule:** The model with the highest validation F1 is selected. If the stacked ensemble outperforms the tuned XGBoost by less than 0.5 percentage points, the simpler tuned XGBoost is preferred on grounds of parsimony (Occam's razor) — greater complexity is only justified by meaningful improvement.

The selected model is referred to as `best_model` throughout the serialisation section.

In [33]:
print("=" * 65)
print(" Model Comparison — Validation Set Results")
print("=" * 65)

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values("F1 (macro)", ascending=False).reset_index(drop=True)

# Add CV result for tuned XGBoost
cv_label = f"{cv_mean:.4f} ± {cv_std:.4f}"
comparison_df["CV F1 (5-fold)"] = comparison_df["Model"].apply(
    lambda m: cv_label if "Tuned" in m else None
)

print(comparison_df.to_string(index=False))

comparison_df.to_csv("artefacts/model_comparison.csv", index=False)
print("\nComparison table saved to artefacts/model_comparison.csv")

# --- Model selection logic ---
best_f1_row   = comparison_df.iloc[0]
tuned_f1_row  = comparison_df[comparison_df["Model"].str.contains("Tuned")].iloc[0]
ensemble_f1_row = comparison_df[comparison_df["Model"].str.contains("Stacked")]

if not ensemble_f1_row.empty:
    delta = ensemble_f1_row.iloc[0]["F1 (macro)"] - tuned_f1_row["F1 (macro)"]
    if delta >= 0.005:
        SELECTED_MODEL_NAME = "Stacked Ensemble"
        best_model = None   # ensemble uses predict_fn — set in serialisation
        print(f"\nSelected: Stacked Ensemble (Δ={delta:+.4f} over tuned XGBoost — exceeds 0.5% threshold)")
    else:
        SELECTED_MODEL_NAME = "Tuned XGBoost"
        best_model = xgb_tuned
        print(f"\nSelected: Tuned XGBoost (ensemble Δ={delta:+.4f} — below 0.5% threshold; prefer parsimony)")
else:
    SELECTED_MODEL_NAME = "Tuned XGBoost"
    best_model = xgb_tuned
    print(f"\nSelected: Tuned XGBoost")

print(f"Selected model : {SELECTED_MODEL_NAME}")
print(f"Validation F1  : {tuned_f1_row['F1 (macro)']:.4f}")
print(f"CV F1 (5-fold) : {cv_mean:.4f} ± {cv_std:.4f}")

 Model Comparison — Validation Set Results
                                             Model  F1 (macro)  Precision (macro)  Recall (macro)  Train Time (s)  CV F1 (5-fold)
         XGBoost Tuned (best random search params)      0.7618             0.7623          0.7614          116.50 0.7606 ± 0.0002
  Stacked XGBoost Ensemble (3 base + cuML LR meta)      0.7524             0.7529          0.7521            1.95            None
             XGBoost Initial (full 7.7M, gpu_hist)      0.7493             0.7498          0.7490           29.60            None
Heterogeneous Stack (XGB + RF + LR → cuML LR meta)      0.7449             0.7454          0.7446            0.08            None
      XGBoost Baseline (200k, DataAnalysis params)      0.7168             0.7169          0.7168            5.70            None
          cuML Random Forest (100 trees, depth 16)      0.7102             0.7122          0.7097           53.90            None

Comparison table saved to artefacts/model_comp

---

## 15. Model Serialisation

All artefacts required by `ModelEvaluation.ipynb` are serialised to the `artefacts/` directory. The design principle is that ModelEvaluation can load these files and run independently without re-training.

| Artefact | Format | Description |
|----------|--------|-------------|
| `best_model.json` | XGBoost native | Preserves `gpu_hist` config and tree structure |
| `scaler.joblib` | joblib | Fitted `StandardScaler` (train set only) |
| `meta_scaler.joblib` | joblib | Scaler for stacking meta-features |
| `meta_model.joblib` | joblib | LogisticRegression meta-model |
| `base_models/base_{i}.json` | XGBoost native | 3 base models for ensemble inference |
| `test_data.npz` | NumPy compressed | **LOCKED test set** — for ModelEvaluation only |
| `val_data.npz` | NumPy compressed | Validation set (reference) |
| `feature_list.pkl` | pickle | `SELECTED_FEATURES` list (27 features) |
| `training_metadata.pkl` | pickle | Hyperparams, CV scores, split sizes, selected model |
| `model_comparison.csv` | CSV | Full comparison table |
| `cv_results.csv` | CSV | Per-fold CV scores |
| `tuning_results.csv` | CSV | All 12 random search trial results |

> `test_data.npz` is locked — it must only be loaded in `ModelEvaluation.ipynb` for the final evaluation step.

In [34]:
print("=" * 55)
print(" Serialisation — Saving All Artefacts")
print("=" * 55)

# 1. Best model (XGBoost native format)
if SELECTED_MODEL_NAME == "Tuned XGBoost":
    xgb_tuned.save_model("artefacts/best_model.json")
    print("Saved: artefacts/best_model.json (Tuned XGBoost)")
else:
    # Stacked ensemble: save component models
    print("Stacked Ensemble selected — base models already saved below.")

# 2. Scaler
joblib.dump(scaler, "artefacts/scaler.joblib")
print("Saved: artefacts/scaler.joblib")

# 3. Stacking components
joblib.dump(meta_scaler, "artefacts/meta_scaler.joblib")
joblib.dump(meta_model,  "artefacts/meta_model.joblib")
print("Saved: artefacts/meta_scaler.joblib")
print("Saved: artefacts/meta_model.joblib")

# 4. Retrain and save each base model on full training set
print("\nRefitting base models on full training set for serialisation...")
for i, config in enumerate(base_configs):
    bm = XGBClassifier(
        tree_method="gpu_hist", eval_metric="logloss",
        random_state=RANDOM_STATE, verbosity=0, **config
    )
    bm.fit(X_train_np, y_train_np, verbose=0)
    bm.save_model(f"artefacts/base_models/base_{i}.json")
    print(f"  Saved: artefacts/base_models/base_{i}.json")
    del bm
    cp.get_default_memory_pool().free_all_blocks()

# 5. Test set — LOCKED
np.savez(
    "artefacts/test_data.npz",
    X_test=X_test_np, y_test=y_test_np
)
print("\nSaved: artefacts/test_data.npz  [LOCKED — for ModelEvaluation only]")

# 6. Validation set
np.savez(
    "artefacts/val_data.npz",
    X_val=X_val_np, y_val=y_val_np
)
print("Saved: artefacts/val_data.npz")

# 7. Feature list
with open("artefacts/feature_list.pkl", "wb") as f:
    pickle.dump(SELECTED_FEATURES, f)
print("Saved: artefacts/feature_list.pkl")

# 8. Training metadata
metadata = {
    "selected_model":    SELECTED_MODEL_NAME,
    "best_params":       best_params,
    "cv_scores":         cv_scores,
    "cv_mean_f1":        float(cv_mean),
    "cv_std_f1":         float(cv_std),
    "val_f1_tuned_xgb":  float(xgb_tuned_result["F1 (macro)"]),
    "val_f1_ensemble":   float(ensemble_f1),
    "train_size":        len(X_train_np),
    "val_size":          len(X_val_np),
    "test_size":         len(X_test_np),
    "selected_features": SELECTED_FEATURES,
    "n_features":        len(SELECTED_FEATURES),
    "random_state":      RANDOM_STATE,
    "split_ratio":       "70/15/15",
}
with open("artefacts/training_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Saved: artefacts/training_metadata.pkl")

print("\n" + "=" * 55)
print(" Serialisation Complete")
print("=" * 55)
print("All artefacts written to artefacts/")
print("Ready for ModelEvaluation.ipynb")

 Serialisation — Saving All Artefacts
Saved: artefacts/best_model.json (Tuned XGBoost)
Saved: artefacts/scaler.joblib
Saved: artefacts/meta_scaler.joblib
Saved: artefacts/meta_model.joblib

Refitting base models on full training set for serialisation...
  Saved: artefacts/base_models/base_0.json
  Saved: artefacts/base_models/base_1.json

Saved: artefacts/test_data.npz  [LOCKED — for ModelEvaluation only]
Saved: artefacts/val_data.npz
Saved: artefacts/feature_list.pkl
Saved: artefacts/training_metadata.pkl

 Serialisation Complete
All artefacts written to artefacts/
Ready for ModelEvaluation.ipynb


---

## 16. Discussion & Justification

### Results Summary

| Model | Val F1 | Val Precision | Val Recall | Train Time |
|-------|--------|--------------|-----------|------------|
| XGBoost Baseline (200k) | 0.7168 | 0.7169 | 0.7168 | ~6s |
| XGBoost Initial (7.7M, gpu_hist) | 0.7493 | 0.7498 | 0.7490 | ~30s |
| cuML Random Forest (100 trees) | 0.7102 | 0.7122 | 0.7097 | ~54s |
| Homogeneous Stack (XGB×2 + LR meta) | 0.7524 | 0.7529 | 0.7521 | — |
| Heterogeneous Stack (XGB+RF+LR + LR meta) | 0.7449 | 0.7454 | 0.7446 | — |
| **XGBoost Tuned (random search best)** | **0.7618** | **0.7623** | **0.7614** | ~117s |

CV F1 (5-fold, tuned XGBoost): **0.7606 ± 0.0002**

---

### Model Selection Rationale

XGBoost was identified as the primary model family in DataAnalysis due to its superior F1 score (0.738 vs cuML RF 0.699) on a 200k sample. This advantage is consistent with the theoretical properties of gradient boosting on structured tabular data with non-linear feature interactions. The HIGGS dataset's kinematic features — particularly the engineered invariant masses — encode complex physics relationships that benefit from the sequential error-correction mechanism of boosting.

Scaling to the full 7.7M training set (F1=0.7493) confirmed this advantage. Hyperparameter tuning via random search further improved performance to F1=0.7618 (best config: depth=12, lr=0.05, 800 trees). The cross-validated F1 of 0.7606 ± 0.0002 demonstrates tight generalisation with extremely low variance across folds — the model is stable and not overfitting to any particular data partition.

### Metric Justification

Macro F1-score was chosen as the primary evaluation metric throughout. For near-balanced binary classification (53/47 class ratio), macro F1 is appropriate because:
- It equally weights precision and recall, penalising models that sacrifice one for the other
- It treats both classes symmetrically, not assuming signal or background is more important
- In the physics context, both false positives (background misidentified as signal) and false negatives (Higgs events missed) have scientific consequences — an accuracy-only metric would mask the error structure between classes

### Hyperparameter Tuning Justification

Random search  was preferred over grid search for the following reasons:
- The hyperparameter space has high dimensionality (8 parameters, ~25,000 possible grid points)
- Not all parameters contribute equally to performance; random search allocates sampling budget across all dimensions independently rather than enumerating all combinations
- Computational constraints on 7.7M rows (each trial ~30–120s on GPU) favour 12 well-distributed random samples over exhaustive enumeration
- The best configuration found (F1=0.7618) was consistent across the top-5 trials, suggesting the search adequately explored the high-performance region

### Stacking Analysis

Two stacking architectures were evaluated, both using a cuML LogisticRegression meta-learner trained on out-of-fold (OOF) predictions to prevent data leakage (Wolpert, 1992).

**Homogeneous stack (XGBoost × 2):** F1=0.7524 — improved over the shallow XGBoost base model (depth=4, F1≈0.73) but fell below the tuned XGBoost. Error correlation within the same algorithm family limits the meta-learner's ability to recover from systematic misclassifications: when both base models agree incorrectly, the meta-model has no signal to correct them.

**Heterogeneous stack (XGBoost + RF + LR):** F1=0.7449 — the lowest ensemble result. Algorithmic diversity introduced the RF and LR as base models, but both perform considerably below the XGBoost in isolation (RF≈0.71, LR<0.70). Ensemble diversity requires not only different algorithms but adequate *base model strength* — weak base models introduce noise that the meta-learner cannot compensate for, regardless of algorithmic variety. This is consistent with the theoretical condition for stacking benefit: base model predictions must be individually informative for the meta-learner to combine them effectively.

**Conclusion:** Neither ensemble configuration exceeded the single tuned XGBoost by more than 0.5 percentage points. Applying Occam's razor, the tuned XGBoost is selected as the final model: higher validation F1 (0.7618), stable CV F1 (0.7606 ± 0.0002), and no ensemble complexity overhead at inference time.

### GPU Memory Management

Training at 7.7M rows imposes non-trivial GPU memory constraints (Quadro RTX 8000, 48 GB). Key management decisions:
- **cuDF → numpy conversion** before stratified splitting: cuML `StratifiedKFold` is unreliable at this scale; sklearn requires CPU arrays
- **`del` + `free_all_blocks()` + `gc.collect()`** after every OOF fold: each XGBClassifier or cuML RF fold run allocates ~1–3 GB; without explicit cleanup the 15-fold het stacking loop would OOM at ~fold 8
- **Base model 3 removal** from the homogeneous stack: depth=12, 800 trees on 6.1M fold rows exceeded available VRAM; reduced to depth=8, 500 trees. This practical constraint informed the decision to use lighter cuML models in the het stack.

### Pipeline Modularity

This notebook deliberately defers all test-set evaluation to `ModelEvaluation.ipynb`. This three-notebook separation follows strict separation of concerns:
1. **DataAnalysis:** Understand the data — no modelling decisions
2. **ModelTraining:** Build and select models — validation set only, test set locked
3. **ModelEvaluation:** Report final performance — no retraining, no model selection

This prevents the academic pitfall of "peeking" at the test set during model selection, which inflates reported performance and undermines the validity of the final evaluation. The test set has not been accessed at any point in this notebook.

---


---

## 17. Summary & Handover to ModelEvaluation.ipynb

### What This Notebook Achieved

This notebook completed the second stage of a three-part ML pipeline for HIGGS boson classification. Starting from 11 million simulated collision events and 27 features selected in DataAnalysis, the following steps were carried out:

1. **Data preparation** — Stratified 70/15/15 split preserving the 53/47 class ratio; StandardScaler fitted on training data only; cuDF-to-numpy conversion for sklearn compatibility.

2. **Model training and comparison** — Five configurations evaluated on the validation set:

| Model | Val F1 |
|-------|--------|
| XGBoost Baseline (200k, DataAnalysis params) | 0.7168 |
| XGBoost Initial (full 7.7M, gpu_hist) | 0.7493 |
| cuML Random Forest (100 trees, depth 16) | 0.7102 |
| Homogeneous Stack (XGBoost × 2 + LR meta) | 0.7524 |
| Heterogeneous Stack (XGB + RF + LR + LR meta) | 0.7449 |
| **XGBoost Tuned (random search, depth=12, lr=0.05)** | **0.7618** |

3. **Hyperparameter tuning** — Random search over 12 trials; best configuration retrained on full training set with early stopping.

4. **Cross-validation** — 5-fold stratified CV on training set only: **F1 = 0.7606 ± 0.0002** — confirms the validation estimate is stable and not a lucky artefact of the particular val split.

5. **Model selection** — Tuned XGBoost selected; no ensemble exceeded it by the required 0.5% threshold to justify additional complexity.

6. **Serialisation** — All artefacts written to `artefacts/`.

---

### Artefacts Produced

| File | Description | Consumed by |
|------|-------------|-------------|
| `artefacts/best_model.json` | Tuned XGBoost (XGBoost native format) | ModelEvaluation |
| `artefacts/scaler.joblib` | StandardScaler fitted on train set | ModelEvaluation |
| `artefacts/meta_scaler.joblib` | Scaler for homogeneous stacking OOF | ModelEvaluation |
| `artefacts/meta_model.joblib` | Homogeneous LR meta-model | ModelEvaluation |
| `artefacts/meta_scaler_het.joblib` | Scaler for heterogeneous stacking OOF | ModelEvaluation |
| `artefacts/meta_model_het.joblib` | Heterogeneous LR meta-model | ModelEvaluation |
| `artefacts/base_models/base_{i}.json` | Homogeneous XGBoost base models | ModelEvaluation |
| `artefacts/base_models/het_base_0.json` | Het base: XGBoost | ModelEvaluation |
| `artefacts/base_models/het_base_1.joblib` | Het base: cuML RF | ModelEvaluation |
| `artefacts/base_models/het_base_2.joblib` | Het base: cuML LR | ModelEvaluation |
| `artefacts/test_data.npz` | **LOCKED test set** (X_test, y_test) | ModelEvaluation only |
| `artefacts/val_data.npz` | Validation set (X_val, y_val) | ModelEvaluation (reference) |
| `artefacts/feature_list.pkl` | 27 selected feature names | ModelEvaluation |
| `artefacts/training_metadata.pkl` | Hyperparams, CV scores, split sizes | ModelEvaluation |
| `artefacts/model_comparison.csv` | Full validation comparison table | Reporting |
| `artefacts/cv_results.csv` | Per-fold CV F1 scores | Reporting |
| `artefacts/tuning_results.csv` | All 12 random search trial results | Reporting |

---

### What ModelEvaluation.ipynb Will Do

The final notebook unlocks the held-out test set (1.65M rows, ~15% of data) for the first time. No model retraining or hyperparameter changes are permitted at that stage. The evaluation will include:

- **Test-set predictions** using `best_model.json` (Tuned XGBoost)
- **Classification report** — per-class precision, recall, F1
- **Confusion matrix** — visualised as a heatmap; absolute counts and normalised rates
- **ROC curve and AUC** — threshold-independent discrimination ability
- **Precision-Recall curve** — especially informative for the signal class
- **Comparison of validation vs test F1** — checks for overfitting to the validation split
- **Final conclusions** — model strengths, limitations, and discussion of the physics classification task

>  **Critical constraint for ModelEvaluation.ipynb:** Load `test_data.npz` exactly once, at the start. Do not use the test set for any intermediate decisions, threshold selection, or model comparison. All decisions in this notebook were made on `X_val_np` only — the test set is a single-use final report.